# Full Monte Carlo Fantasy Point Scoring

The analytic prototype (`probabilistic_scoring.ipynb`) treated each raw prediction as the centre of a
normal distribution and *integrated* to get expected points. That only smooths the point estimate, and
it lost to the ranked lookup on MAE (12.40 vs 10.91). It also couldn't express the real prize:
a full **distribution** of outcomes per driver.

This notebook builds the full Monte Carlo simulation from the V3 backlog:

1. **Noisy grids** - perturb the raw XGBoost outputs with noise sampled from *bucketed empirical
   residual distributions* (tiers by predicted position), then **rank each simulation** to a valid grid.
2. **Discrete events** - DNF as `Bernoulli(dnf_rate)`, one Fastest Lap winner and one Driver-of-the-Day
   winner per sim (weighted draws), overtakes as `Poisson(expected)`.
3. **Real scoring** - score every simulated race through the actual `score_driver_race` /
   `score_driver_qualifying` functions, producing a **points distribution** per driver.
4. **Evaluate** - does the MC mean beat the ranked lookup on MAE? And, more importantly, are the
   **P10-P90 bands calibrated** (do ~80% of actual outcomes land inside them)? Calibration is the
   payoff even if the point estimate doesn't move.

Unlike the analytic version, DNF is modelled by *re-ranking survivors ahead of retirees* each sim, so a
front-runner's DNF correctly promotes everyone behind - the interaction the closed-form couldn't capture.

**Scope note:** sprint scoring is not simulated here (sprint weekends are flagged and reported
separately). Quali and finish noise are sampled independently, which slightly inflates positions-gained
variance - flagged in the calibration section.

## Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fastf1

from app.config import (
    PROCESSED_HISTORIC_FEATURES_DIR, PROCESSED_PRACTICE_FEATURES_DIR,
    PROCESSED_CIRCUIT_FEATURES_DIR, INTERIM_EVENTS_DIR, INTERIM_RACES_DIR,
    INTERIM_QUALI_DIR, INTERIM_SPRINT_QUALIFYING_DIR, FANTASY_POINTS_DIR,
    TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS,
)
from app.models.configs import QUALI_POSITION_MODEL, FINISH_POSITION_MODEL
from app.models.predict import load_season_model
from app.data.overtakes import build_overtake_predictor
from app.data.dotd import build_dotd_predictor
from app.data.scoring_rules import (
    DRIVER_RACE_POSITION_POINTS, DRIVER_QUALI_POSITION_POINTS,
    FASTEST_LAP_POINTS, DOTD_POINTS, POSITION_GAINED_POINTS,
    RACE_PENALTY, OVERTAKE_MADE_POINTS,
)
from app.models.compose import FASTEST_LAP_PROB

predict_overtakes = build_overtake_predictor()
predict_dotd = build_dotd_predictor()

RNG = np.random.default_rng(20260804)
N_SIMS = 1000

# scoring lookup tables as dense arrays (index = position). sized with headroom
# because a few races ran more than 20 entries (reserve/substitute drivers); any
# position past the points-paying slots simply scores 0.
GRID = 41
RACE_PTS = np.zeros(GRID)
QUALI_PTS = np.zeros(GRID)
for p, v in DRIVER_RACE_POSITION_POINTS.items():
    RACE_PTS[p] = v
for p, v in DRIVER_QUALI_POSITION_POINTS.items():
    QUALI_PTS[p] = v
FL_PROB_ARR = np.zeros(GRID)
for p, v in FASTEST_LAP_PROB.items():
    FL_PROB_ARR[p] = v
print('setup ok')

## Collect raw predictions and actuals

Same as the prototype: run the prediction pipeline but keep the raw (pre-ranking) XGBoost outputs
alongside the ranked positions, actual positions, and actual fantasy points.

In [ ]:
def predict_with_raw(quali_model, quali_config, finish_model, finish_config, season, round_num):
    """same as predict() but also returns the raw (pre-ranking) XGBoost outputs."""
    historic_features = pd.read_parquet(PROCESSED_HISTORIC_FEATURES_DIR / f"{season}_{round_num:02d}.parquet")

    practice_path = PROCESSED_PRACTICE_FEATURES_DIR / f"{season}_{round_num:02d}.parquet"
    practice_features = pd.read_parquet(practice_path) if practice_path.exists() else None

    if practice_features is not None:
        features = historic_features.merge(practice_features, on=["race_id", "driver_id"], how="left")
    else:
        features = historic_features.copy()
        practice_cols = set(c for c in quali_config["features"] + finish_config["features"] if c.startswith("fp"))
        for col in practice_cols:
            features[col] = float("nan")

    circuit_path = PROCESSED_CIRCUIT_FEATURES_DIR / f"{season}_{round_num:02d}.parquet"
    if circuit_path.exists():
        circuit_features = pd.read_parquet(circuit_path)
        features = features.merge(circuit_features, on="race_id", how="left")
    else:
        circuit_cols = set(c for c in quali_config["features"] + finish_config["features"] if c.startswith("circuit_"))
        for col in circuit_cols:
            features[col] = float("nan")

    sq_path = INTERIM_SPRINT_QUALIFYING_DIR / f"{season}_{round_num:02d}.parquet"
    if sq_path.exists():
        sq = pd.read_parquet(sq_path).set_index("driver_id")["sprint_quali_position"]
        features["sprint_quali_position"] = features["driver_id"].map(sq)
    else:
        features["sprint_quali_position"] = float("nan")

    for col in set(quali_config["features"] + finish_config["features"]):
        if col not in features.columns:
            features[col] = float("nan")

    X_quali = features[quali_config["features"]]
    raw_quali = quali_model.predict(X_quali)
    features["predicted_quali_position"] = pd.Series(raw_quali).rank(method="first").astype(int).values

    X_finish = features[finish_config["features"]]
    raw_finish = finish_model.predict(X_finish)
    features["predicted_finish_position"] = pd.Series(raw_finish).rank(method="first").astype(int).values

    return pd.DataFrame({
        "driver_id": features["driver_id"],
        "constructor_id": features["constructor_id"],
        "predicted_quali_position": features["predicted_quali_position"],
        "predicted_finish_position": features["predicted_finish_position"],
        "raw_quali": raw_quali,
        "raw_finish": raw_finish,
        "sprint_quali_position": features["sprint_quali_position"],
        "rolling_crash_dnf_rate_last_5": features["rolling_crash_dnf_rate_last_5"] if "rolling_crash_dnf_rate_last_5" in features.columns else 0.0,
    }).sort_values("predicted_finish_position").reset_index(drop=True)

In [ ]:
rows = []

for season in [2023, 2024, 2025, 2026]:
    quali_model = load_season_model(QUALI_POSITION_MODEL, season)
    finish_model = load_season_model(FINISH_POSITION_MODEL, season)
    schedule = fastf1.get_event_schedule(season)
    schedule = schedule[schedule["RoundNumber"] > 0]

    for _, event in schedule.iterrows():
        rnd = event["RoundNumber"]
        fp = PROCESSED_HISTORIC_FEATURES_DIR / f"{season}_{rnd:02d}.parquet"
        tp = FANTASY_POINTS_DIR / f"{season}_{rnd:02d}.csv"
        if not (fp.exists() and tp.exists()):
            continue

        ep = INTERIM_EVENTS_DIR / f"{season}_{rnd:02d}.parquet"
        location = pd.read_parquet(ep)["location"].iloc[0] if ep.exists() else None

        preds = predict_with_raw(quali_model, QUALI_POSITION_MODEL, finish_model, FINISH_POSITION_MODEL, season, rnd)

        targets = pd.read_csv(tp)
        driver_targets = targets[targets["asset_type"] == "driver"].set_index("asset_id")

        races_path = INTERIM_RACES_DIR / f"{season}_{rnd:02d}.parquet"
        quali_path = INTERIM_QUALI_DIR / f"{season}_{rnd:02d}.parquet"
        if not (races_path.exists() and quali_path.exists()):
            continue
        race_results = pd.read_parquet(races_path).set_index("driver_id")
        quali_results = pd.read_parquet(quali_path).set_index("driver_id")

        is_sprint = bool(preds["sprint_quali_position"].notna().any())

        for _, row in preds.iterrows():
            did = row["driver_id"]
            if did not in driver_targets.index or did not in race_results.index:
                continue
            actual_finish = race_results.loc[did, "finish_position"]
            actual_quali = quali_results.loc[did, "quali_position"] if did in quali_results.index else np.nan
            if pd.isna(actual_finish) or pd.isna(actual_quali):
                continue

            rows.append({
                "season": season, "round": rnd, "driver_id": did,
                "constructor_id": row["constructor_id"],
                "raw_finish": row["raw_finish"], "raw_quali": row["raw_quali"],
                "ranked_finish": row["predicted_finish_position"],
                "ranked_quali": row["predicted_quali_position"],
                "actual_finish": int(actual_finish), "actual_quali": int(actual_quali),
                "actual_fantasy_pts": driver_targets.loc[did, "fantasy_points"],
                "dnf_rate": row["rolling_crash_dnf_rate_last_5"],
                "dnf_flag": bool(race_results.loc[did, "dnf_flag"]),
                "location": location, "is_sprint": is_sprint,
            })

    print(f"  {season}: done")

data = pd.DataFrame(rows)
data["raw_finish_residual"] = data["raw_finish"] - data["actual_finish"]
data["raw_quali_residual"] = data["raw_quali"] - data["actual_quali"]
print(f"\ntotal: {len(data)} driver-race records, seasons {sorted(data['season'].unique())}")
data.head()

## Calibrate bucketed residual distributions

The V3 spec calls for *bucketed residual distributions* rather than a single Gaussian sigma. We bin by
**predicted-position tier** (the only thing known at sim time) and, within each tier, keep the empirical
pool of residuals `raw - actual`. Sampling from these pools preserves skew that a normal can't:
a predicted front-runner can drop a long way back but can't finish better than P1, so its residual pool
is asymmetric.

Calibrated on `TRAIN_SEASONS + 2023` (held out from the 2024-2026 evaluation).

In [ ]:
cal = data[data["season"].isin(list(TRAIN_SEASONS) + [2023])].copy()

# predicted-position tiers (bucket on the raw prediction, since that's all we know at sim time)
TIER_EDGES = [0.0, 3.5, 7.5, 12.5, 21.0]
TIER_LABELS = ["P1-3", "P4-7", "P8-12", "P13-20"]

def tier_of(raw):
    return np.clip(np.digitize(raw, TIER_EDGES[1:-1]), 0, 3)

# residual pools: {tier_index: np.array of residuals}
finish_pools = {}
quali_pools = {}
for t in range(4):
    finish_pools[t] = cal.loc[tier_of(cal["raw_finish"].values) == t, "raw_finish_residual"].dropna().values
    quali_pools[t] = cal.loc[tier_of(cal["raw_quali"].values) == t, "raw_quali_residual"].dropna().values

# per-tier mean/std for the correlated-position factor decomposition (preserves marginal variance)
FIN_MEAN = {t: float(finish_pools[t].mean()) for t in range(4)}
FIN_STD = {t: float(finish_pools[t].std()) for t in range(4)}
QUA_MEAN = {t: float(quali_pools[t].mean()) for t in range(4)}
QUA_STD = {t: float(quali_pools[t].std()) for t in range(4)}

print("finish residual pools (raw - actual):")
for t in range(4):
    p = finish_pools[t]
    print(f"  {TIER_LABELS[t]:>7}: n={len(p):4d}  mean={p.mean():+.2f}  std={p.std():.2f}  "
          f"P10={np.percentile(p,10):+.1f}  P90={np.percentile(p,90):+.1f}")
print("quali residual pools:")
for t in range(4):
    p = quali_pools[t]
    print(f"  {TIER_LABELS[t]:>7}: n={len(p):4d}  mean={p.mean():+.2f}  std={p.std():.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
for t in range(4):
    axes[t].hist(finish_pools[t], bins=20, color="#3498db", alpha=0.8, edgecolor="black")
    axes[t].axvline(0, color="black", ls="--")
    axes[t].set_title(f"finish residuals {TIER_LABELS[t]}")
    axes[t].set_xlabel("raw - actual")
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()

## Calibrate DNF probability by predicted position

The first pass sampled DNF from `rolling_crash_dnf_rate_last_5` (mean ~0.05) - about a third of the true
~13-15% retirement rate, and crash-only. Worse, the crash/mechanical split is unreliable from 2024 on
(FastF1 reports a generic "Retired"), so those rolling features are shaky exactly in the evaluation window.

The reliable signal is the plain `dnf_flag` (status-based), and in the data DNFs are overwhelmingly a
back-of-grid event: ~47% retirement rate among actual P16+ finishers, ~0% in the top ten. So we calibrate
`P(DNF | predicted finish tier)` empirically - a pace proxy that's reliable across all seasons and gives
every driver a nonzero, position-scaled retirement chance. This is what should fatten the downside tail and
stop MC over-rating crash-prone value drivers.

In [ ]:
# calibrate DNF probability as a function of PREDICTED finish position (pace proxy).
# uses the reliable status-based dnf_flag, not the crash/mechanical split.
POS_BINS = [0, 5, 10, 15, GRID]
POS_LABELS = ["pred P1-5", "pred P6-10", "pred P11-15", "pred P16+"]
cal_bin = np.clip(np.digitize(cal["ranked_finish"].values, POS_BINS[1:-1]), 0, 3)

dnf_by_bin = np.array([cal.loc[cal_bin == b, "dnf_flag"].mean() for b in range(4)])
print("P(DNF) by predicted finish tier (calibration seasons):")
for b in range(4):
    print(f"  {POS_LABELS[b]:>13}: {dnf_by_bin[b]:.3f}  (n={int((cal_bin==b).sum())})")

# expand to a per-position lookup (index = predicted finish position 1..GRID-1)
DNF_PROB_BY_POS = np.zeros(GRID)
for pos in range(1, GRID):
    DNF_PROB_BY_POS[pos] = dnf_by_bin[np.clip(np.digitize(pos, POS_BINS[1:-1]), 0, 3)]

# sanity: expected DNFs per 20-car grid vs the observed ~2.5
print(f"\nexpected DNFs per 20-car grid: {DNF_PROB_BY_POS[1:21].sum():.2f}  (observed ~2.5)")
print(f"old sampler mean prob:        {np.nan_to_num(cal['dnf_rate']).mean():.3f}  (crash-only, too low)")

## Calibrate the correlated-chaos (frailty) shock

The sim so far draws every driver's DNF independently, which can never produce "several cars out in the
same race". Real races are bimodal: lots of clean 0-2 DNF races and a fat tail of 5-6 DNF carnage races.
We add one shared per-race multiplier `z ~ Gamma(mean 1, var phi)` applied to every driver's DNF odds at
once - a shared-frailty model. It preserves each driver's marginal rate (E[z]=1) but couples them, so a
"bad race" sim retires many cars together. We fit the single parameter `phi` to the observed
over-dispersion of DNFs-per-race (its variance beyond what independent draws would give).

In [ ]:
# fit frailty over-dispersion phi so the SIMULATED DNF-count variance matches the observed variance
# (the moment estimate under-shoots because independent binomial var < mean; search on the real target)
dnf_counts = cal.groupby(["season", "round"])["dnf_flag"].sum()
mean_N, var_N = dnf_counts.mean(), dnf_counts.var()
probs20 = DNF_PROB_BY_POS[1:21]   # representative full 20-car grid
_rng = np.random.default_rng(0)

def _sim_dnf_var(phi, n=60000):
    z = np.ones(n) if phi <= 0 else _rng.gamma(1.0 / phi, phi, n)
    return float(((_rng.random((n, 20)) < np.clip(probs20[None, :] * z[:, None], 0, 1)).sum(axis=1)).var())

grid = np.linspace(0.0, 0.6, 61)
FRAILTY_PHI = float(min(grid, key=lambda p: abs(_sim_dnf_var(p) - var_N)))
print(f"DNFs per race: mean {mean_N:.2f}, observed var {var_N:.2f}")
print(f"independent (phi=0) simulated var {_sim_dnf_var(0.0):.2f}")
print(f"=> fitted frailty phi = {FRAILTY_PHI:.3f}  (simulated var {_sim_dnf_var(FRAILTY_PHI):.2f}, target {var_N:.2f})")

## Monte Carlo engine

Vectorised over all `N_SIMS` at once for each race. Per simulation:

- `latent = raw - residual_sample`, so lower latent = better. Sampled residuals come from each driver's
  predicted tier pool.
- **DNF** is drawn `Bernoulli(P(DNF | predicted position))` from the calibrated curve above, so back-of-grid
  value drivers carry their true retirement risk.
- **DNF-aware finish grid:** sort by `latent + BIG * dnf` so survivors take positions 1..k in latent order
  and retirees fill k+1..20 (worst latent classified lowest). This promotes drivers behind a DNF.
- **Quali grid:** rank latent quali (qualifying happens before the race, so DNF doesn't reshuffle it).
- **Fastest lap:** one winner per sim, a weighted draw among survivors by `FASTEST_LAP_PROB[finish_pos]`
  (Gumbel-max trick for a vectorised weighted choice).
- **DOTD:** one winner per sim, weighted by the model's per-driver DOTD probability.
- **Overtakes:** `Poisson(expected)` per driver, expected from the overtake predictor at the ranked quali slot.
- Score every driver every sim with the real scoring tables (DNF penalty replaces position points but FL /
  DOTD / overtakes still count, matching `score_driver_race`).

In [ ]:
def _ranks(x):
    """dense competition-free ranks 1..n along axis=1 (argsort of argsort)."""
    return np.argsort(np.argsort(x, axis=1), axis=1) + 1

def run_race_mc(race, n_sims=N_SIMS, rng=RNG, frailty_phi=0.0, rho=0.0):
    """returns per-driver simulated total-points matrix (n_sims x n_drivers) and the driver order.

    frailty_phi > 0 adds a shared per-race Gamma(mean 1, var phi) multiplier on every driver's DNF odds,
    correlating retirements. rho > 0 adds a shared per-race position-scramble factor via a variance-
    preserving decomposition (each driver's residual keeps its marginal mean/var but gains correlation rho
    with the rest of the grid). Both 0 recovers the independent model."""
    race = race.reset_index(drop=True)
    n = len(race)
    raw_f = race["raw_finish"].values
    raw_q = race["raw_quali"].values
    # DNF probability from the calibrated predicted-position curve (pace proxy)
    dnf_rate = DNF_PROB_BY_POS[np.clip(race["ranked_finish"].values, 1, GRID - 1)]
    tf = tier_of(raw_f)
    tq = tier_of(raw_q)

    # sample residuals from each driver's tier pool -> (n_sims, n)
    resid_f = np.empty((n_sims, n))
    resid_q = np.empty((n_sims, n))
    # shared per-race position-scramble factors (one draw per sim, common to all drivers)
    g_f = rng.standard_normal(n_sims) if rho > 0 else None
    g_q = rng.standard_normal(n_sims) if rho > 0 else None
    for i in range(n):
        bf = rng.choice(finish_pools[tf[i]], size=n_sims)
        bq = rng.choice(quali_pools[tq[i]], size=n_sims)
        if rho > 0:
            # variance-preserving blend: keeps marginal mean/var, injects correlation rho via g
            mf, sf = FIN_MEAN[tf[i]], FIN_STD[tf[i]]
            mq, sq = QUA_MEAN[tq[i]], QUA_STD[tq[i]]
            resid_f[:, i] = mf + np.sqrt(1 - rho) * (bf - mf) + np.sqrt(rho) * sf * g_f
            resid_q[:, i] = mq + np.sqrt(1 - rho) * (bq - mq) + np.sqrt(rho) * sq * g_q
        else:
            resid_f[:, i] = bf
            resid_q[:, i] = bq

    latent_f = raw_f[None, :] - resid_f
    latent_q = raw_q[None, :] - resid_q

    if frailty_phi > 0:
        z = rng.gamma(1.0 / frailty_phi, frailty_phi, size=n_sims)   # shared per-race shock, mean 1
        dnf_rate_sim = np.clip(dnf_rate[None, :] * z[:, None], 0.0, 1.0)
    else:
        dnf_rate_sim = np.broadcast_to(dnf_rate[None, :], (n_sims, n))
    dnf = rng.random((n_sims, n)) < dnf_rate_sim

    # DNF-aware finish ranking: survivors first (small key), retirees pushed to the back
    BIG = 1e6
    finish_pos = _ranks(latent_f + BIG * dnf)
    quali_pos = _ranks(latent_q)
    positions_gained = quali_pos - finish_pos

    # fastest lap: weighted single winner among survivors
    fl_w = FL_PROB_ARR[finish_pos] * (~dnf)
    g = -np.log(-np.log(rng.random((n_sims, n)) + 1e-12) + 1e-12)
    fl_winner = np.argmax(np.log(fl_w + 1e-12) + g, axis=1)
    fl_flag = np.zeros((n_sims, n), dtype=bool)
    fl_flag[np.arange(n_sims), fl_winner] = True

    # driver of the day: weighted single winner across the field
    dotd_p = np.asarray(predict_dotd(race["driver_id"])).astype(float)
    dotd_p = dotd_p / dotd_p.sum() if dotd_p.sum() > 0 else np.full(n, 1.0 / n)
    g2 = -np.log(-np.log(rng.random((n_sims, n)) + 1e-12) + 1e-12)
    dotd_winner = np.argmax(np.log(dotd_p[None, :] + 1e-12) + g2, axis=1)
    dotd_flag = np.zeros((n_sims, n), dtype=bool)
    dotd_flag[np.arange(n_sims), dotd_winner] = True

    # overtakes: Poisson around the predictor's expectation at the ranked quali slot
    exp_ot = np.array([
        predict_overtakes(race.loc[i, "driver_id"], race.loc[i, "location"],
                          int(race.loc[i, "season"]), int(race.loc[i, "ranked_quali"]))
        for i in range(n)
    ])
    exp_ot = np.clip(np.nan_to_num(exp_ot, nan=0.0), 0, None)
    overtakes = rng.poisson(exp_ot[None, :] * np.ones((n_sims, 1)))

    # score through the real tables
    race_base = np.where(dnf, RACE_PENALTY, RACE_PTS[finish_pos] + positions_gained * POSITION_GAINED_POINTS)
    race_total = (race_base + fl_flag * FASTEST_LAP_POINTS + dotd_flag * DOTD_POINTS
                  + overtakes * OVERTAKE_MADE_POINTS)
    quali_total = QUALI_PTS[quali_pos]
    total = quali_total + race_total  # (n_sims, n)
    return total, race["driver_id"].values

## Run the simulation across the evaluation seasons

2024-2026. For each race we run `N_SIMS` simulations and collapse each driver's distribution to
mean / median / P10 / P90 / std. The ranked point estimate is computed alongside for a like-for-like
MAE comparison against the prototype's baseline.

In [ ]:
ranked_score = (
    QUALI_PTS[data["ranked_quali"].clip(1, 20)]
    + RACE_PTS[data["ranked_finish"].clip(1, 20)]
    + (data["ranked_quali"] - data["ranked_finish"]) * POSITION_GAINED_POINTS
)
data["ranked_score"] = ranked_score

eval_races = data[data["season"].isin([2024, 2025, 2026])].groupby(["season", "round"])

records = []
for (season, rnd), race in eval_races:
    race = race.copy()
    race["season"] = season
    totals, driver_ids = run_race_mc(race)
    mean = totals.mean(axis=0)
    median = np.median(totals, axis=0)
    p10 = np.percentile(totals, 10, axis=0)
    p90 = np.percentile(totals, 90, axis=0)
    std = totals.std(axis=0)
    r = race.reset_index(drop=True)
    for i, did in enumerate(driver_ids):
        records.append({
            "season": season, "round": rnd, "driver_id": did,
            "constructor_id": r.loc[i, "constructor_id"],
            "mc_mean": mean[i], "mc_median": median[i], "mc_p10": p10[i], "mc_p90": p90[i], "mc_std": std[i],
            "ranked_score": r.loc[i, "ranked_score"],
            "actual_fantasy_pts": r.loc[i, "actual_fantasy_pts"],
            "actual_finish": r.loc[i, "actual_finish"],
            "is_sprint": r.loc[i, "is_sprint"],
        })

mc = pd.DataFrame(records)
print(f"simulated {mc.groupby(['season','round']).ngroups} races, {len(mc)} driver-races, {N_SIMS} sims each")
mc.head()

## Evaluation 1 - point-estimate accuracy

Does the MC mean beat the ranked lookup on MAE? The prototype's analytic version could not (12.40 vs
10.91). Sprint weekends are excluded from the headline number since MC doesn't score sprints.

In [ ]:
ev = mc[~mc["is_sprint"]].copy()
ev["mc_err"] = ev["mc_mean"] - ev["actual_fantasy_pts"]
ev["ranked_err"] = ev["ranked_score"] - ev["actual_fantasy_pts"]

print("=== OVERALL (non-sprint) ===")
print(f"MC mean MAE:   {ev['mc_err'].abs().mean():.2f}   bias {ev['mc_err'].mean():+.2f}")
print(f"ranked MAE:    {ev['ranked_err'].abs().mean():.2f}   bias {ev['ranked_err'].mean():+.2f}")

print("\n=== BY ACTUAL FINISH TIER ===")
ev["tier"] = pd.cut(ev["actual_finish"], bins=[0,3,6,10,15,22], labels=["P1-3","P4-6","P7-10","P11-15","P16+"])
cmp = ev.groupby("tier", observed=True).agg(
    n=("driver_id","count"),
    mc_mae=("mc_err", lambda x: x.abs().mean()),
    ranked_mae=("ranked_err", lambda x: x.abs().mean()),
    mc_bias=("mc_err","mean"),
    ranked_bias=("ranked_err","mean"),
).round(1)
cmp["mc_gain"] = (cmp["ranked_mae"] - cmp["mc_mae"]).round(1)
print(cmp.to_string())

print("\n=== BY SEASON ===")
for s in sorted(ev["season"].unique()):
    sub = ev[ev["season"] == s]
    print(f"  {s}: MC MAE={sub['mc_err'].abs().mean():.1f}  ranked MAE={sub['ranked_err'].abs().mean():.1f}  "
          f"MC bias={sub['mc_err'].mean():+.1f}  ranked bias={sub['ranked_err'].mean():+.1f}")

## Evaluation 2 - distribution calibration (the real payoff)

Even if the mean doesn't beat the ranked lookup, the MC output is only *useful* if its uncertainty bands
are honest. Two checks:

- **P10-P90 coverage:** the fraction of actual outcomes falling inside the 80% band should be ~0.80.
- **Reliability curve:** across many nominal quantiles, does the empirical hit rate track the diagonal?

If the band is well calibrated, the optimiser can trust `mc_p10` / `mc_p25` as a downside signal for
risk-aware selection - something the ranked point estimate simply cannot provide.

In [ ]:
cov = mc[~mc["is_sprint"]].copy()
inside = ((cov["actual_fantasy_pts"] >= cov["mc_p10"]) & (cov["actual_fantasy_pts"] <= cov["mc_p90"]))
print(f"P10-P90 coverage: {inside.mean():.1%}  (target 80%)")
print(f"below P10:        {(cov['actual_fantasy_pts'] < cov['mc_p10']).mean():.1%}  (target 10%)")
print(f"above P90:        {(cov['actual_fantasy_pts'] > cov['mc_p90']).mean():.1%}  (target 10%)")

# reliability curve: need per-driver-race quantile of the actual within its sim distribution.
# recompute quantile of actual vs each driver's sim draws by re-simulating quantile levels.
qlevels = np.array([0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95])
# approximate the actual's percentile position using the stored summary is not enough;
# instead compute empirical coverage at each nominal level from p-bands we can derive per level.
# rebuild by re-running MC and recording each actual's rank - do it compactly here.
hit = {q: [] for q in qlevels}
for (season, rnd), race in data[data["season"].isin([2024,2025,2026])].groupby(["season","round"]):
    race = race.copy(); race["season"] = season
    if race["is_sprint"].any():
        continue
    totals, dids = run_race_mc(race)
    r = race.reset_index(drop=True)
    for i, did in enumerate(dids):
        actual = r.loc[i, "actual_fantasy_pts"]
        draws = totals[:, i]
        for q in qlevels:
            hit[q].append(actual <= np.quantile(draws, q))
emp = np.array([np.mean(hit[q]) for q in qlevels])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot([0,1],[0,1],"k--",lw=1,label="perfect")
axes[0].plot(qlevels, emp, "o-", color="#e74c3c", label="MC")
axes[0].set_xlabel("nominal quantile"); axes[0].set_ylabel("empirical coverage")
axes[0].set_title("reliability curve"); axes[0].legend()

axes[1].hist(cov["mc_std"], bins=30, color="#9b59b6", alpha=0.8, edgecolor="black")
axes[1].set_xlabel("per-driver simulated std (points)"); axes[1].set_ylabel("count")
axes[1].set_title("spread of predicted uncertainty")
plt.tight_layout(); plt.show()

## Does correlated chaos fix the JOINT (team-level) risk?

Per-driver bands can each be honest while the *team* band is too tight - independent sampling never lets
several of your drivers fail in the same race, so a team's downside is understated. That is the number
the optimiser's risk-aware objective would rely on. Test whether the actual **constructor** total (2
correlated team-mates) and a random **5-driver team** total land inside their P10-P90 ~80% of the time,
independent (phi=0) vs the frailty shock. Per-driver (marginal) coverage should barely move; the joint
coverage should rise toward 80%.

In [ ]:
from collections import defaultdict

def joint_coverage(phi, rho, n_teams=100, seed=7):
    rng2 = np.random.default_rng(seed)
    marg_in = marg_n = con_in = con_n = team_in = team_n = 0
    eval_mask = data["season"].isin([2024, 2025, 2026]) & ~data["is_sprint"]
    for (s, rnd), race in data[eval_mask].groupby(["season", "round"]):
        race = race.reset_index(drop=True)
        T, dids = run_race_mc(race, frailty_phi=phi, rho=rho)
        dids = list(dids)
        col = {d: i for i, d in enumerate(dids)}
        actual = dict(zip(race["driver_id"], race["actual_fantasy_pts"]))
        con = dict(zip(race["driver_id"], race["constructor_id"]))
        lo = np.percentile(T, 10, axis=0); hi = np.percentile(T, 90, axis=0)
        for d in dids:
            marg_n += 1; marg_in += int(lo[col[d]] <= actual[d] <= hi[col[d]])
        cmap = defaultdict(list)
        for d in dids: cmap[con[d]].append(d)
        for c, mem in cmap.items():
            if len(mem) < 2: continue
            dist = T[:, [col[d] for d in mem]].sum(axis=1)
            act = sum(actual[d] for d in mem)
            con_n += 1; con_in += int(np.percentile(dist, 10) <= act <= np.percentile(dist, 90))
        if len(dids) >= 5:
            for _ in range(n_teams):
                idx = rng2.choice(len(dids), 5, replace=False)
                dist = T[:, idx].sum(axis=1)
                act = sum(actual[dids[j]] for j in idx)
                team_n += 1; team_in += int(np.percentile(dist, 10) <= act <= np.percentile(dist, 90))
    return marg_in / marg_n, con_in / con_n, team_in / team_n

print(f"{'model':>22} {'per-driver':>11} {'constructor':>12} {'5-driver team':>14}   (target 80%)")
for phi, rho, label in [(0.0, 0.0, "independent"),
                        (FRAILTY_PHI, 0.0, "frailty DNF only"),
                        (FRAILTY_PHI, 0.15, "+ position corr .15"),
                        (FRAILTY_PHI, 0.30, "+ position corr .30"),
                        (FRAILTY_PHI, 0.50, "+ position corr .50")]:
    m, c, t = joint_coverage(phi, rho)
    print(f"{label:>22} {m:11.1%} {c:12.1%} {t:14.1%}")

## Example race - per-driver distributions

What a single race looks like: predicted median with an 80% band and the realised outcome. This is the
view that would drive the dashboard.

In [ ]:
SEASON, ROUND = 2026, 5
race = data[(data["season"] == SEASON) & (data["round"] == ROUND)].copy()
race["season"] = SEASON
totals, dids = run_race_mc(race)
r = race.reset_index(drop=True)

order = np.argsort(-totals.mean(axis=0))
fig, ax = plt.subplots(figsize=(11, 7))
for y, i in enumerate(order):
    d = totals[:, i]
    lo, med, hi = np.percentile(d, [10, 50, 90])
    ax.plot([lo, hi], [y, y], color="#3498db", lw=3, alpha=0.6)
    ax.plot(med, y, "o", color="#2c3e50", markersize=6)
    ax.plot(r.loc[i, "actual_fantasy_pts"], y, "*", color="#e74c3c", markersize=14)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([dids[i] for i in order], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("fantasy points")
ax.set_title(f"{SEASON} Round {ROUND}: MC median + P10-P90 band (blue) vs actual (red star)")
ax.axvline(0, color="gray", ls=":", lw=1)
plt.tight_layout(); plt.show()

print(f"{'driver':<24}{'P10':>7}{'median':>8}{'mean':>7}{'P90':>7}{'std':>7}{'actual':>8}")
for i in order:
    d = totals[:, i]
    print(f"{dids[i]:<24}{np.percentile(d,10):>7.1f}{np.median(d):>8.1f}{d.mean():>7.1f}"
          f"{np.percentile(d,90):>7.1f}{d.std():>7.1f}{r.loc[i,'actual_fantasy_pts']:>8.0f}")

## Constructor distributions and downside risk

Summing the two drivers' simulated points per sim gives a constructor points distribution (quali bonus
and pitstop points are deterministic add-ons and omitted here for the shape comparison). The gap between
`mean` and `P25` is a concrete **downside-risk** measure the optimiser could penalise - the risk-aware
objective from the V3 backlog.

In [ ]:
race = data[(data["season"] == SEASON) & (data["round"] == ROUND)].copy()
race["season"] = SEASON
totals, dids = run_race_mc(race)
cons = pd.DataFrame(totals, columns=race.reset_index(drop=True)["constructor_id"].values)
cons_tot = cons.T.groupby(level=0).sum().T  # sum both drivers per constructor per sim

summary = pd.DataFrame({
    "mean": cons_tot.mean(),
    "p25": cons_tot.quantile(0.25),
    "median": cons_tot.median(),
    "p75": cons_tot.quantile(0.75),
    "std": cons_tot.std(),
}).sort_values("mean", ascending=False)
summary["downside(mean-p25)"] = (summary["mean"] - summary["p25"]).round(1)
print(summary.round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
top = summary.head(6).index
for c in top:
    ax.hist(cons_tot[c], bins=30, alpha=0.5, label=c)
ax.set_xlabel("constructor fantasy points (both drivers)")
ax.set_ylabel("count"); ax.set_title(f"{SEASON} Round {ROUND}: constructor points distributions")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Verdict and next steps

Read the two evaluations together:

- **Point estimate (Eval 1):** if `MC mean MAE` is within a point or so of the ranked baseline, the
  simulation isn't *costing* accuracy - which is the bar, since MC exists for the distribution, not a
  better mean.
- **Calibration (Eval 2):** if P10-P90 coverage is near 80% and the reliability curve hugs the diagonal,
  the uncertainty is trustworthy and the payoff is real.

If both hold, the wiring is:

1. Add an MC scorer alongside `compose_drivers` that returns `mc_mean`, `mc_p10`, `mc_p25`, `mc_p90`, `mc_std`.
2. Pass the bands to the dashboard (median + P10-P90 per driver, constructor histograms).
3. Extend the optimiser with a risk-aware objective (maximise `p25`, or `mean - lambda * std`) and backtest
   it against the current expected-value selection.

If calibration is off, the fixes are in this notebook's knobs: widen/narrow the residual pools, add
quali-finish correlation (currently independent, which inflates positions-gained variance), or refine the
DNF-rate source.